# Channel labels — offline acceptance

All settings use temporary local files. The API and dashboard tests send no instrument commands. The final cell runs the real dashboard JavaScript against a small DOM double; it checks interaction and text handling, not browser layout. Each case is directly visible below.

In [ ]:
import tempfile
from pathlib import Path
from malab.instruments.ad5780 import AD5780
from malab.instruments.dac_array import DACArray
from malab.instruments.channel_labels import LabelStore
from malab.instruments.resistive_load import ResistorStore
from malab.dashboard.controller import Dashboard

temporary=tempfile.TemporaryDirectory();base=Path(temporary.name)
labels=LabelStore(base/'labels.local.json');resistors=ResistorStore(base/'resistors.local.json')
def rejects(function):
    try:function()
    except ValueError:return
    raise AssertionError('Expected ValueError')

array=DACArray(['192.0.2.10','192.0.2.20'],label_store=labels,resistor_store=resistors)
assert array.get_label(6)==''
assert array.set_label(6,'  Bias coil  ')== 'Bias coil'
assert array.get_label(6)=='Bias coil'
assert array.devices[1].get_label(2)=='Bias coil'
single=AD5780(address='192.0.2.20:23',enabled=False,label_store=LabelStore(labels.path))
assert single.get_label(2)=='Bias coil'
reordered=DACArray(['192.0.2.20','192.0.2.10'],label_store=LabelStore(labels.path))
assert reordered.get_label(2)=='Bias coil' and reordered.get_label(6)==''
array.add_resistor(6,2000)
assert array.set_label(6,'Cooling Ω')=='Cooling Ω'
assert array.get_resistance(6)==2000
assert all(not d.connected for d in array.devices)
print('PASS: set/get, restart persistence, shared single/array API, identity across reorder, resistor unchanged; no connections.')

In [ ]:
for invalid in [None,False,123,'x'*81,'bad\nlabel','bad\x00label']:
    rejects(lambda value=invalid:array.set_label(6,value))
for invalid in [0,9,True,2.5]:rejects(lambda channel=invalid:array.set_label(channel,'test'))
assert array.get_label(6)=='Cooling Ω'
assert array.set_label(6,'a'*80)=='a'*80
assert array.set_label(6,'  ')=='' and array.get_label(6)==''
# HTML-like text remains text; it is not sent to an instrument or interpreted.
assert array.set_label(6,'<b>Bias</b>')=='<b>Bias</b>'
print('PASS: bounded one-line text, invalid inputs, Unicode and clearing.')

In [ ]:
class NoIO:
    def __init__(self,**kwargs):self.bench_port=None;self.calls=[]
    def close(self):pass

d=Dashboard(['192.0.2.10','192.0.2.20'],meter_address='192.0.2.71',meter_channel=6,
            dac_factory=NoIO,label_store=labels,resistor_store=resistors)
u=d.units[1];u.state['network']['state']='disconnected';u.state['ramp']['state']='RUNNING'
request=dict(action='label',unit=2,channel=6,label='Heater bias',request_id='rename')
job=d.submit(request)
assert job['state']=='complete' and job['channel']==6
assert d.submit(request)==job
rejects(lambda:d.submit(dict(request,label='Changed retry')))
rejects(lambda:d.submit(dict(request,request_id='wrong-unit',unit=1)))
s=d.snapshot();assert s['channels'][5]['label']=='Heater bias'
assert all(not row['label'] for row in s['channels'][:5]+s['channels'][6:])
assert s['channels'][5]['resistance_ohms']==2000
assert u.state['network']['state']=='disconnected' and u.state['ramp']['state']=='RUNNING'
assert all(not unit.dac.calls and unit.commands.empty() for unit in d.units)
assert all(unit.state['pending'] is None for unit in d.units)
d.submit(dict(action='label',channel=6,label='',request_id='reset'))
assert d.snapshot()['channels'][5]['label']==''
d.close();array.close();reordered.close();single.close();temporary.cleanup()
print('PASS: dashboard saves/resets while disconnected or ramping without affecting operations, routing, resistors or output queues.')

In [ ]:
import subprocess
script=r"""
const fs=require('fs'),vm=require('vm'),assert=require('node:assert/strict');
const elements=new Map();
class Element{
 constructor(){this.value='';this.children=[];this.attrs={};this.hidden=false;this.disabled=false;this.validation='';}
 set innerHTML(value){this.html=value;for(const m of value.matchAll(/<[^>]+\bid="([^"]+)"[^>]*>/g)){const el=new Element();for(const a of m[0].matchAll(/([\w-]+)="([^"]*)"/g))el[a[1]]=a[2];elements.set(m[1],el);}}
 get innerHTML(){return this.html||'';}
 append(...children){this.children.push(...children);}
 replaceChildren(...children){this.children=children;}
 setAttribute(k,v){this.attrs[k]=v;}
 removeAttribute(k){delete this[k];}
 focus(){}
 setCustomValidity(value){this.validation=value;}
 reportValidity(){if(this.type==='text')return this.value.length<=80;return !this.validation&&this.value!==''&&Number.isFinite(Number(this.value))&&(this.min==null||Number(this.value)>=Number(this.min))&&(this.max==null||Number(this.value)<=Number(this.max));}
}
const body=new Element();body.innerHTML=fs.readFileSync('malab/dashboard/static/index.html','utf8');
const context=vm.createContext({console,performance,AbortSignal,crypto:require('node:crypto').webcrypto,
 document:{getElementById:id=>elements.get(id)||null,createElement:()=>new Element(),createTextNode:text=>({text}),querySelector:()=>({content:'offline-token'})},
 setInterval:()=>1,setTimeout:()=>1,fetch:()=>new Promise(()=>{}),location:{reload:()=>{throw Error('Unexpected reload');}},captures:[]});
vm.runInContext(fs.readFileSync('malab/dashboard/static/app.js','utf8'),context);
const now=Date.now()/1000;
context.fixture={now,server_started:1,channel_count:4,units:[{unit:1,address:'192.0.2.10',first_channel:1,last_channel:4,usb_port:null,network:{state:'good',last_ok:now,latency_ms:1},recovery:{state:'unsupported'},ramp:{state:'IDLE'},pending:null,device_error:null}],
 channels:Array.from({length:4},(_,i)=>({channel:i+1,local_channel:i+1,unit:1,voltage:1,at:now,source:'readback',limits:[-10,10],calibrated:false,resistance_ohms:null,current_mA:null,current_limits_mA:null})),
 meter:{channel:2,address:'192.0.2.71',state:'good',at:now,voltage:1.01,resistance_ohms:null,current_mA:null},jobs:{},events:[],history:[{at:now,channels:[1,1,1,1],currents:[null,.5,null,null],resistances:[null,2000,null,null],meter:1.01,meter_current:.505}]};
const run=code=>vm.runInContext(code,context),el=id=>elements.get(id);
run('state=fixture;received=performance.now();build();render();drawTrace();send=(unit,data)=>captures.push(data);');

assert.equal(el('channel-label-2').textContent,'DAC 2');
el('rename-2').onclick();assert.equal(el('label-form-2').hidden,false);assert.equal(el('label-input-2').value,'');
el('label-input-2').value='Bias coil';el('label-form-2').onsubmit({preventDefault(){}});
assert.equal(context.captures.at(-1).action,'label');assert.equal(context.captures.at(-1).channel,2);assert.equal(context.captures.at(-1).label,'Bias coil');
run("state.channels[1].label='Bias coil';render();drawTrace();");
assert.equal(el('channel-label-2').textContent,'Bias coil');assert.match(el('meter-heading').textContent,/Bias coil \(DAC 2\)/);
assert.ok(el('legend').children.some(c=>c.textContent==='Bias coil (DAC 2)'));
el('label-input-2').value='Unsaved edit';run('render();');assert.equal(el('label-input-2').value,'Unsaved edit');
el('cancel-label-2').onclick();assert.equal(el('label-form-2').hidden,true);
el('rename-2').onclick();assert.equal(el('label-input-2').value,'Bias coil');
run(`state.channels[1].label='<img src=x onerror=alert(1)>';render();drawTrace();`);
assert.equal(el('channel-label-2').textContent,'<img src=x onerror=alert(1)>');assert.equal(el('channel-label-2').innerHTML,'');
assert.match(el('meter-heading').textContent,/<img/);
el('clear-label-2').onclick();assert.equal(context.captures.at(-1).label,'');
run("state.channels[1].label='';render();drawTrace();");assert.equal(el('channel-label-2').textContent,'DAC 2');
run('received=performance.now()-3000;render();');assert.equal(el('save-label-2').disabled,true);
console.log('PASS: Rename/Save/Cancel/Reset, meter/history names, preserved drafts, safe text rendering and stale-control disabling (DOM double).');
"""
result=subprocess.run(["node","-e",script],capture_output=True,text=True,encoding="utf-8")
print(result.stdout)
assert result.returncode==0,result.stderr